In [2]:
import nltk
import math
from collections import Counter

nltk.download('brown')
nltk.download('punkt')

from nltk.corpus import brown
from nltk.tokenize import word_tokenize

text = brown.raw()

# Tokenization
words = word_tokenize(text.lower())
words = words[:5000]
unigrams = words
bigrams = list(zip(words[:-1], words[1:]))
trigrams = list(zip(words[:-2], words[1:-1], words[2:]))
unigram_count = Counter(unigrams)
bigram_count = Counter(bigrams)
trigram_count = Counter(trigrams)


[nltk_data] Downloading package brown to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [3]:
vocab = set(words)
V = len(vocab)

N = len(words)

print("Vocabulary Size:", V)
print("Total Words:", N)

def unigram_mle(word):
    return unigram_count[word] / N


def bigram_mle(w1, w2):
    if unigram_count[w1] == 0:
        return 0
    return bigram_count[(w1, w2)] / unigram_count[w1]


def trigram_mle(w1, w2, w3):
    if bigram_count[(w1, w2)] == 0:
        return 0
    return trigram_count[(w1, w2, w3)] / bigram_count[(w1, w2)]


Vocabulary Size: 1438
Total Words: 5000


In [4]:

def unigram_laplace(word):
    return (unigram_count[word] + 1) / (N + V)


def bigram_laplace(w1, w2):
    return (bigram_count[(w1, w2)] + 1) / (unigram_count[w1] + V)


def trigram_laplace(w1, w2, w3):
    return (trigram_count[(w1, w2, w3)] + 1) / (bigram_count[(w1, w2)] + V)


In [5]:

def perplexity_unigram(test_words, smoothing=True):
    log_prob = 0
    count = 0

    for word in test_words:
        if smoothing:
            p = unigram_laplace(word)
        else:
            p = unigram_mle(word)

        if p > 0:
            log_prob += math.log(p)
            count += 1

    return math.exp(-log_prob / count)


def perplexity_bigram(test_words, smoothing=True):
    log_prob = 0
    count = 0

    for i in range(1, len(test_words)):
        w1 = test_words[i - 1]
        w2 = test_words[i]

        if smoothing:
            p = bigram_laplace(w1, w2)
        else:
            p = bigram_mle(w1, w2)

        if p > 0:
            log_prob += math.log(p)
            count += 1

    return math.exp(-log_prob / count)


def perplexity_trigram(test_words, smoothing=True):
    log_prob = 0
    count = 0

    for i in range(2, len(test_words)):
        w1 = test_words[i - 2]
        w2 = test_words[i - 1]
        w3 = test_words[i]

        if smoothing:
            p = trigram_laplace(w1, w2, w3)
        else:
            p = trigram_mle(w1, w2, w3)

        if p > 0:
            log_prob += math.log(p)
            count += 1

    return math.exp(-log_prob / count)


In [6]:

test_words = words[4500:5000]

print("\nPerplexity with MLE:")

print("Unigram:", perplexity_unigram(test_words, False))
print("Bigram :", perplexity_bigram(test_words, False))
print("Trigram:", perplexity_trigram(test_words, False))

print("\nPerplexity with Laplace Smoothing:")

print("Unigram:", perplexity_unigram(test_words, True))
print("Bigram :", perplexity_bigram(test_words, True))
print("Trigram:", perplexity_trigram(test_words, True))


print("\nExample MLE Probabilities:")

print("P(the) =", unigram_mle("the"))
print("P(is | this) =", bigram_mle("this", "is"))
print("P(on | is, the) =", trigram_mle("is", "the", "on"))

print("\nExample Laplace Probabilities:")

print("P(the) =", unigram_laplace("the"))
print("P(is | this) =", bigram_laplace("this", "is"))
print("P(on | is, the) =", trigram_laplace("is", "the", "on"))


Perplexity with MLE:
Unigram: 295.43216227076493
Bigram : 5.47002210768698
Trigram: 1.8583523234117691

Perplexity with Laplace Smoothing:
Unigram: 306.88123684006996
Bigram : 302.5084851731645
Trigram: 484.9105831867996

Example MLE Probabilities:
P(the) = 0.0
P(is | this) = 0
P(on | is, the) = 0

Example Laplace Probabilities:
P(the) = 0.00015532774153463808
P(is | this) = 0.0006954102920723226
P(on | is, the) = 0.0006954102920723226
